# Orientation & Spatial Metadata: 
##### `nnUNet assumes spatial coherence — it will not correct this`

1. All images and labels in same orientation (RAS) `this has been checked and fixed code in D:\projects\4D-flow-MRI-aortic-segmentation-\preprocessing\check_fix_orientation.ipynb`
2. Matching:
    * shape
    * spacing
    * affine

In [ ]:
import os
import numpy as np
import nibabel as nib
from scipy.ndimage import label as cc_label
from collections import Counter, defaultdict

# ========================
# PATHS
# ========================
imgs_dir = r"D:\projects\4D-flow-MRI-aortic-segmentation-\data\ao_imgs"
segs_dir = r"D:\projects\4D-flow-MRI-aortic-segmentation-\data\ao_segs"

# ========================
# PARAMETERS
# ========================
SPACING_TOL = 1e-4
MIN_LABEL_VOXELS = 1000      # adjust based on anatomy size
MAX_CC = 3                  # max allowed connected components
BORDER_MARGIN = 1            # voxels from edge considered "touching"

# ========================
# STORAGE
# ========================
summary = Counter()
issues = defaultdict(list)
label_volumes = []

# ========================
# HELPERS
# ========================
def touches_border(mask, margin=1):
    return (
        mask[:margin, :, :].any() or mask[-margin:, :, :].any() or
        mask[:, :margin, :].any() or mask[:, -margin:, :].any() or
        mask[:, :, :margin].any() or mask[:, :, -margin:].any()
    )

# ========================
# MAIN LOOP
# ========================
for lbl_name in os.listdir(segs_dir):
    if not lbl_name.endswith(".nii.gz"):
        continue

    img_name = lbl_name.replace(".nii.gz", "_0000.nii.gz")
    img_path = os.path.join(imgs_dir, img_name)
    lbl_path = os.path.join(segs_dir, lbl_name)

    if not os.path.exists(img_path):
        issues["missing_image"].append(lbl_name)
        continue

    img = nib.load(img_path)
    lbl = nib.load(lbl_path)

    img_data = img.get_fdata()
    lbl_data = lbl.get_fdata()

    case_id = lbl_name.replace(".nii.gz", "")

    # ---- Orientation
    img_orient = nib.aff2axcodes(img.affine)
    lbl_orient = nib.aff2axcodes(lbl.affine)

    if img_orient != ('R','A','S'):
        summary["img_non_RAS"] += 1
        issues["img_non_RAS"].append(case_id)

    if lbl_orient != ('R','A','S'):
        summary["lbl_non_RAS"] += 1
        issues["lbl_non_RAS"].append(case_id)

    if img_orient != lbl_orient:
        issues["img_lbl_orient_mismatch"].append(case_id)

    # ---- Shape & spacing
    if img.shape != lbl.shape:
        issues["shape_mismatch"].append(case_id)
    #get the voxel sizes in millimeters
    if not np.allclose(img.header.get_zooms()[:3],
                       lbl.header.get_zooms()[:3],
                       atol=SPACING_TOL):
        issues["spacing_mismatch"].append(case_id)

    # ---- Image intensity sanity
    if not np.isfinite(img_data).all():
        issues["non_finite_intensity"].append(case_id)

    if img_data.std() < 1e-6:
        issues["near_constant_image"].append(case_id)

    # ---- Label checks
    lbl_bin = lbl_data > 0
    lbl_voxels = int(lbl_bin.sum())
    label_volumes.append(lbl_voxels)

    if lbl_voxels < MIN_LABEL_VOXELS:
        issues["very_small_label"].append(case_id)

    if lbl_voxels == 0:
        issues["empty_label"].append(case_id)

    if touches_border(lbl_bin, BORDER_MARGIN):
        issues["label_touches_border"].append(case_id)

    # ---- Connected components
    cc, num_cc = cc_label(lbl_bin)
    if num_cc > MAX_CC:
        issues["too_many_components"].append(case_id)

    summary["cases_checked"] += 1

# ========================
# REPORT
# ========================
print("\n========== QA SUMMARY ==========")
for k, v in summary.items():
    print(f"{k:25s}: {v}")

print("\n========== ISSUE DETAILS ==========")
for issue, cases in issues.items():
    print(f"\n{issue} ({len(cases)} cases)")
    for c in cases[:10]:
        print("  ", c)
    if len(cases) > 10:
        print("  ...")

# ========================
# VOLUME OUTLIERS
# ========================
if label_volumes:
    vols = np.array(label_volumes)
    low, high = np.percentile(vols, [5, 95])

    print("\n========== LABEL VOLUME STATS ==========")
    print(f"Min   : {vols.min()}")
    print(f"5%    : {low}")
    print(f"Median: {np.median(vols)}")
    print(f"95%   : {high}")
    print(f"Max   : {vols.max()}")


========== QA SUMMARY ==========
cases_checked            : 744

========== ISSUE DETAILS ==========

very_small_label (1 cases)
   AORTA_29265

empty_label (1 cases)
   AORTA_29265

========== LABEL VOLUME STATS ==========
Min   : 0
5%    : 7890.9
Median: 16164.5
95%   : 30835.3
Max   : 53742


In [ ]:
import matplotlib.pyplot as plt

# ========================
# PATHS
# ========================
imgs_dir = r"D:\projects\4D-flow-MRI-aortic-segmentation-\data\ao_imgs"
segs_dir = r"D:\projects\4D-flow-MRI-aortic-segmentation-\data\ao_segs"

case_id = "AORTA_29265"

img_path = os.path.join(imgs_dir, f"{case_id}_0000.nii.gz")
lbl_path = os.path.join(segs_dir, f"{case_id}.nii.gz")

# ========================
# LOAD DATA
# ========================
img = nib.load(img_path)
lbl = nib.load(lbl_path)

img_data = img.get_fdata()
lbl_data = lbl.get_fdata()

# ========================
# PRINT METADATA
# ========================
print("=== METADATA ===")
print("Image orientation:", nib.aff2axcodes(img.affine))
print("Label orientation:", nib.aff2axcodes(lbl.affine))
print("Image shape:", img.shape)
print("Label shape:", lbl.shape)
print("Image spacing:", img.header.get_zooms()[:3])
print("Label spacing:", lbl.header.get_zooms()[:3])
print("Label voxel count:", int((lbl_data > 0).sum()))
print("================\n")

# ========================
# SLICE SELECTION
# ========================
mask = lbl_data > 0


=== METADATA ===
Image orientation: ('R', 'A', 'S')
Label orientation: ('R', 'A', 'S')
Image shape: (72, 144, 240)
Label shape: (72, 144, 240)
Image spacing: (np.float32(1.6000035), np.float32(2.083333), np.float32(2.0833333))
Label spacing: (np.float32(1.6000035), np.float32(2.083333), np.float32(2.0833333))
Label voxel count: 0



: 